In [4]:
# Install packages (in a notebook cell)
%pip install tensorflow==2.11 keras==2.11
%pip install -U albumentations
%pip install segmentation-models==1.0.1  # Ensure compatibility with TensorFlow 2.11
%pip install matplotlib

# Imports
import tensorflow as tf
from tensorflow.keras import backend as K  # ✅ safer than using `import keras.backend`

import cv2
import matplotlib.pyplot as plt
import numpy as np

import albumentations as A
import segmentation_models as sm

# Set segmentation_models framework
sm.set_framework('tf.keras')
sm.framework()  # Ensure the framework is properly initialized

%matplotlib widget  # ✅ Works best inside of VS Code

  Using cached tensorflow-2.11.0-cp310-cp310-win_amd64.whl (1.9 kB)
  Using cached keras-2.11.0-py2.py3-none-any.whl (1.7 MB)
  Using cached tensorflow_intel-2.11.0-cp310-cp310-win_amd64.whl (266.3 MB)
  Using cached tensorflow_estimator-2.11.0-py2.py3-none-any.whl (439 kB)
  Using cached tensorboard-2.11.2-py3-none-any.whl (6.0 MB)
  Using cached flatbuffers-25.2.10-py2.py3-none-any.whl (30 kB)
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 1.12
    Uninstalling flatbuffers-1.12:
      Successfully uninstalled flatbuffers-1.12
  Attempting uninstall: tensorflow-estimator
    Found existing installation: tensorflow-estimator 2.9.0
    Uninstalling tensorflow-estimator-2.9.0:
      Successfully uninstalled tensorflow-estimator-2.9.0
  Attempting uninstall: keras
    Found existing installation: keras 2.9.0
    Uninstalling keras-2.9.0:
      Successfully uninstalled keras-2.9.0
  Attempting uninstall: tensorboard
    Found existing installation: tensorb

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\hp\\Downloads\\Erosion-detection\\Erosion-detection-main\\venv\\Lib\\site-packages\\tensorflow\\python\\_pywrap_py_exception_registry.pyd'
Check the permissions.


[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#'

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [ ]:
raster_path = 'T36UXV_20200406T083559_TCI_10m.jp2'
with rasterio.open(raster_path, 'r', driver = 'JP2OpenJPEG') as src:
    raster_img = src.read()
    raster_meta = src.meta

In [ ]:
raster_img = reshape_as_image(raster_img)

plt.figure(figsize=(15,15))
plt.imshow(raster_img)

In [ ]:
train_df = gpd.read_file("masks/Masks_T36UXV_20190427.shp")
print(len(train_df))
train_df.head(5)

In [ ]:
train_df['geometry'][0].exterior.coords.xy

In [ ]:
train_df = gpd.read_file("masks/Masks_T36UXV_20190427.shp")


# let's remove rows without geometry
train_df = train_df[train_df.geometry.notnull()]

# assigning crs
train_df.crs = {'init' :'epsg:4324'}

#transforming polygons to the raster crs
train_df = train_df.to_crs({'init' : raster_meta['crs']['init']})

In [ ]:
# rasterize works with polygons that are in image coordinate system

def poly_from_utm(polygon, transform):
    poly_pts = []
    
    # make a polygon from multipolygon
    poly = cascaded_union(polygon)
    for i in np.array(poly.exterior.coords):
        
        # transfrom polygon to image crs, using raster meta
        poly_pts.append(~transform * tuple(i))
        
    # make a shapely Polygon object
    new_poly = Polygon(poly_pts)
    return new_poly

# creating binary mask for field/not_filed segmentation.

poly_shp = []
im_size = (src.meta['height'], src.meta['width'])
for num, row in train_df.iterrows():
    if row['geometry'].geom_type == 'Polygon':
        poly = poly_from_utm(row['geometry'], src.meta['transform'])
        poly_shp.append(poly)
    else:
        for p in row['geometry']:
            poly = poly_from_utm(p, src.meta['transform'])
            poly_shp.append(poly)

mask = rasterize(shapes=poly_shp,
                 out_shape=im_size)

# plotting the mask

plt.figure(figsize=(15,15))
plt.imshow(mask)

In [ ]:
bin_mask_meta = src.meta.copy()
bin_mask_meta.update({'count': 1})
with rasterio.open('big_mask.jp2", 'w', **bin_mask_meta) as dst:
    dst.write(mask * 255, 1)

# Crop

In [ ]:
raster_path = 'T36UXV_20200406T083559_TCI_10m.jp2'
with rasterio.open(raster_path, 'r', driver = 'JP2OpenJPEG') as src:
    big_img = src.read()
    img_meta = src.meta

raster_path = 'big_mask.jp2'
with rasterio.open(raster_path, 'r', driver = 'JP2OpenJPEG') as src:
    big_mask = src.read()
    mask_meta = src.meta

In [ ]:
big_img = reshape_as_image(big_img)
big_mask = reshape_as_image(big_mask)

c_img_path = "cropped_img_256"
c_mask_path = "cropped_mask_256"

def crop_img(big_img, shape, outfolder):
    i = 0
    for r in range(0,big_img.shape[0],shape[0]):
        for c in range(0,big_img.shape[1],shape[1]):
            img_path = os.path.join(outfolder, f'{i}.png')
            if r+shape[0] <= big_img.shape[0] and c + shape[1] <= big_img.shape[1]:
                cv2.imwrite(img_path, cv2.cvtColor(big_img[r:r+shape[0], c:c+shape[1],:], cv2.COLOR_RGB2BGR))
                i += 1
                
crop_img(big_img, (256, 256), c_img_path)
crop_img(big_mask, (256, 256), c_mask_path)

c_img_path = "cropped_img_512"
c_mask_path = "cropped_mask_512"

crop_img(big_img, (512, 512), c_img_path)
crop_img(big_mask, (512, 512), c_mask_path)

# EDA

In [ ]:
def extract_imgs(TRAIN_DIR, MASK_DIR, train_list, mask_list, mode = 'limited'):
    out_rgb = []
    out_mask = []

    counter_empty = 0

    for p_img, p_mask in zip(train_list, mask_list):   
        img_path = os.path.join(TRAIN_DIR, p_img)
        mask_path = os.path.join(MASK_DIR, p_mask)
    
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB) / 255.
        mask = cv2.imread(mask_path)
    
        mask = mask[:, :, :1]
        mask[mask > 0.] = 1.
        
        if mode == 'limited':
            if 1 not in mask: counter_empty+=1
    
            if not(1 not in mask and counter_empty >= 40):
                out_rgb += [img]
                out_mask += [mask]
        elif mode == 'not_limited':
            if 1 not in mask: counter_empty+=1
            out_rgb += [img]
            out_mask += [mask]
        

    out_rgb = np.array(out_rgb, dtype = 'float32')
    out_mask = np.array(out_mask, dtype = 'float32')
    if mode == 'limited':
        return out_rgb, out_mask
    if mode == 'not_limited':
        return out_rgb, out_mask, counter_empty

BASE_DIR = ''
TRAIN_DIR = BASE_DIR + 'cropped_img_256/'
MASK_DIR = BASE_DIR + 'cropped_mask_256/'

train = os.listdir(TRAIN_DIR)
mask = os.listdir(MASK_DIR)

print(f"Train files: {len(train)}. ---> {train[:3]}")
print(f"Test files :  {len(mask)}. ---> {mask[:3]}")

all_rgb_256, all_mask_256, c_256 = extract_imgs(TRAIN_DIR, MASK_DIR, train, mask, mode = 'not_limited')
rgb_256, mask_256 = extract_imgs(TRAIN_DIR, MASK_DIR, train, mask, mode = 'limited')

TRAIN_DIR = BASE_DIR + 'cropped_img_512/'
MASK_DIR = BASE_DIR + 'cropped_mask_512/'

train = os.listdir(TRAIN_DIR)
mask = os.listdir(MASK_DIR)

print(f"Train files: {len(train)}. ---> {train[:3]}")
print(f"Test files :  {len(mask)}. ---> {mask[:3]}")

all_rgb_512, all_mask_512, c_512 = extract_imgs(TRAIN_DIR, MASK_DIR, train, mask, mode = 'not_limited')
rgb_512, mask_512 = extract_imgs(TRAIN_DIR, MASK_DIR, train, mask, mode = 'limited')

## Pixel ratio

In [ ]:
print(f"All 256: {all_rgb_256.shape[0]}. ---> Empty: {c_256}")
print(f"All 512:  {all_rgb_512.shape[0]}. ---> Empty {c_512}")

In [ ]:
def pixel_ratio(masks):
    ratio = []
    for mask in masks:
        ratio.append(mask[mask>0].shape[0] /  (masks.shape[0]*masks.shape[1]))
    return np.average(ratio)

all_ratio_256 = pixel_ratio(all_mask_256)
ratio_256 = pixel_ratio(mask_256)
all_ratio_512 = pixel_ratio(all_mask_512)
ratio_512 = pixel_ratio(mask_512)

In [ ]:
x = np.array([all_ratio_256,ratio_256, all_ratio_512, ratio_512])
y = np.array([all_ratio_256,ratio_256, all_ratio_512, ratio_512])
xticks = ['All 256','256', 'All 512','512']
plt.xticks(x, xticks)
plt.xlabel('Shape')
plt.ylabel('Ratio')
plt.plot(x, y)
plt.show()

## Masks problem

In [ ]:
rows = 1
columns = 2

images_id = [1, 27, 32]
for i in images_id:
    fig = plt.figure(figsize=(10, 7))
    fig.add_subplot(rows, columns, 1)
    plt.imshow(rgb_256[i])
    plt.axis('off')
    plt.title("Image")
    
    fig.add_subplot(rows, columns, 2)
    plt.imshow(mask_256[i], interpolation=None)
   
    plt.axis('off')
    plt.title("Ground truth")